# Data Merge Master — NASCAR Sponsorship Visibility

Integrates race results, Reddit/Google Trends, YouTube, and news data into a single
analysis-ready master dataset.

**Sponsors**: FedEx, NAPA Auto Parts, McDonald's, Love's Travel Stops  
**Season**: 2024 NASCAR Cup Series (Races 1–36)  
**Output**: `data/processed/master_dataset.csv`

## Step 1: Audit Current Data Formats

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Load all datasets
print('Loading datasets...')
race_df   = pd.read_csv('../data/processed/race_results_2024_clean.csv')
reddit_df = pd.read_csv('../data/processed/reddit_mentions_2024.csv')
youtube_df = pd.read_csv('../data/processed/youtube_engagement_2024.csv')
news_df   = pd.read_csv('../data/processed/news_mentions_2024.csv')

print(f'Race Results:       {len(race_df):,} rows')
print(f'Reddit/Trends:      {len(reddit_df):,} rows')
print(f'YouTube Engagement: {len(youtube_df):,} rows')
print(f'News Mentions:      {len(news_df):,} rows')

Loading datasets...
Race Results:       1,398 rows
Reddit/Trends:      104 rows
YouTube Engagement: 104 rows
News Mentions:      93 rows


In [2]:
def audit_dataset(df, name):
    print(f"\n{'='*50}")
    print(f'DATASET: {name}')
    print(f"{'='*50}")
    print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
    print(f'\nColumn Names and Types:')
    print('-' * 40)
    for col in df.columns:
        dtype = df[col].dtype
        sample = df[col].dropna().iloc[0] if not df[col].dropna().empty else 'N/A'
        print(f'  {col:30} | {str(dtype):10} | Sample: {str(sample)[:25]}')
    print(f'\nMissing Values:')
    missing = df.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0])
    else:
        print('  None')

audit_dataset(race_df,    'Race Results')
audit_dataset(reddit_df,  'Reddit/Google Trends')
audit_dataset(youtube_df, 'YouTube Engagement')
audit_dataset(news_df,    'News Mentions')


DATASET: Race Results
Shape: 1398 rows x 21 columns

Column Names and Types:
----------------------------------------
  year                           | int64      | Sample: 2024
  Race_Number_Raw                | int64      | Sample: -1
  Race_Name                      | str        | Sample: Daytona
  Track_Type                     | str        | Sample: superspeedway
  Finish_Position                | int64      | Sample: 1
  Start_Position                 | int64      | Sample: 19
  Car_Number                     | int64      | Sample: 45
  Driver                         | str        | Sample: Tyler Reddick
  Manufacturer                   | str        | Sample: Toyota
  Team                           | str        | Sample: 23XI Racing
  Laps_Completed                 | int64      | Sample: 60
  Laps_Led                       | int64      | Sample: 1
  Status                         | str        | Sample: running
  Points                         | float64    | Sample: 51.0
  Stage_

## Step 2: Standardize Data Formats

In [3]:
DATE_FORMAT = '%Y-%m-%d'

def standardize_dates(df, date_columns):
    df = df.copy()
    for col in date_columns:
        if col not in df.columns:
            print(f'  Warning: column {col!r} not found')
            continue
        df[col] = pd.to_datetime(df[col], errors='coerce')
        failed = df[col].isna().sum()
        if failed > 0:
            print(f'  Warning: {failed} dates failed to convert in {col!r}')
        df[f'{col}_str'] = df[col].dt.strftime(DATE_FORMAT)
    return df

print('Standardizing dates...')
# race_df has no Race_Date — add approximate dates spread across the season
from datetime import timedelta
season_start = datetime(2024, 2, 18)
season_end   = datetime(2024, 11, 10)
total_days   = (season_end - season_start).days
max_race     = race_df['Race_Number'].max()   # highest numbered race
race_date_map = {
    r: (season_start + timedelta(days=total_days * max(r-1, 0) / max(max_race-1, 1))).strftime(DATE_FORMAT)
    for r in range(-2, max_race + 1)
}
race_df['Race_Date'] = race_df['Race_Number'].map(race_date_map)
race_df = standardize_dates(race_df, ['Race_Date'])
youtube_df = standardize_dates(youtube_df, ['race_date'])
news_df['race_date'] = news_df['race_number'].map(
    {r: (season_start + timedelta(days=total_days * max(r-1,0) / max(max_race-1,1))).strftime(DATE_FORMAT)
     for r in range(1, max_race + 1)}
)
news_df = standardize_dates(news_df, ['race_date'])

print('Date standardization complete.')

Standardizing dates...
Date standardization complete.


In [4]:
def standardize_columns(df, column_mapping):
    df = df.copy()
    df = df.rename(columns=column_mapping)
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    return df

race_columns = {
    'Race_Name':       'race_name',
    'Race_Date':       'race_date',
    'Race_Number':     'race_number',
    'Driver':          'driver',
    'Team':            'team',
    'Primary_Sponsor': 'sponsor',
    'Finish_Position': 'finish_position',
    'Laps_Led':        'laps_led',
}
exposure_columns = {
    'Race_Name':   'race_name',
    'Race_Number': 'race_number',
    'Sponsor':     'sponsor',
}

race_df    = standardize_columns(race_df,    race_columns)
reddit_df  = standardize_columns(reddit_df,  exposure_columns)
youtube_df = standardize_columns(youtube_df, exposure_columns)
news_df    = standardize_columns(news_df,    exposure_columns)

# Fix Reddit race_number 0 → 1  (Daytona period was mapped to pre-season race 0, not race 1)
reddit_df['race_number'] = reddit_df['race_number'].replace(0, 1)
print('Reddit race 0 → 1 fix applied.')

# Fix YouTube race_number -2 → 1  (Daytona 500 lookup produced -2)
youtube_df['race_number'] = youtube_df['race_number'].replace(-2, 1)
print('YouTube race -2 → 1 fix applied.')

print('Column name standardization complete.')

Reddit race 0 → 1 fix applied.
YouTube race -2 → 1 fix applied.
Column name standardization complete.


In [5]:
# Sponsor name standardization — keep full canonical names
SPONSOR_NAME_MAPPING = {
    'FedEx':               'FedEx',
    'Fedex':               'FedEx',
    'FEDEX':               'FedEx',
    'NAPA':                'NAPA Auto Parts',
    'NAPA Auto Parts':     'NAPA Auto Parts',
    'Napa':                'NAPA Auto Parts',
    "McDonald's":          "McDonald's",
    'McDonalds':           "McDonald's",
    "Mcdonald's":          "McDonald's",
    "Love's Travel Stops": "Love's Travel Stops",
    'Loves Travel Stops':  "Love's Travel Stops",
    "Love's":              "Love's Travel Stops",
}

TARGET_SPONSORS = ['FedEx', 'NAPA Auto Parts', "McDonald's", "Love's Travel Stops"]

def standardize_sponsor_names(df, mapping, sponsor_col='sponsor'):
    df = df.copy()
    df[sponsor_col] = df[sponsor_col].replace(mapping)
    unexpected = [s for s in df[sponsor_col].unique() if s not in mapping.values()]
    if unexpected:
        print(f'  Unexpected sponsors: {unexpected}')
    return df

race_df    = standardize_sponsor_names(race_df,    SPONSOR_NAME_MAPPING)
reddit_df  = standardize_sponsor_names(reddit_df,  SPONSOR_NAME_MAPPING)
youtube_df = standardize_sponsor_names(youtube_df, SPONSOR_NAME_MAPPING)
news_df    = standardize_sponsor_names(news_df,    SPONSOR_NAME_MAPPING)

# Filter race results to target sponsors only (removes non-target sponsors)
race_df = race_df[race_df['sponsor'].isin(TARGET_SPONSORS)].copy()
# Keep only main-season races (race_number 1–36)
race_df = race_df[race_df['race_number'].between(1, 36)].copy()

print('Sponsor standardization complete.')
print(f'Race results filtered: {len(race_df)} rows')
print('Sponsors in each dataset:')
for name, df in [('Race', race_df), ('Reddit', reddit_df), ('YouTube', youtube_df), ('News', news_df)]:
    print(f'  {name}: {sorted(df["sponsor"].unique())}')

  Unexpected sponsors: ['Jordan Brand', 'Ally', 'Other/Unknown', 'Freeway Insurance', 'Shell/Pennzoil', 'Monster Energy', 'HendrickCars.com', 'Fastenal', 'Busch Light', 'Sunseeker Resorts', 'Craftsman', 'Dow', 'Rheem', 'Discount Tire', 'Dewalt', 'Liberty University', 'Magical Vacation Planner', 'Menards', "Cheddar's Scratch Kitchen", 'Bass Pro Shops']
Sponsor standardization complete.
Race results filtered: 144 rows
Sponsors in each dataset:
  Race: ['FedEx', "Love's Travel Stops", "McDonald's", 'NAPA Auto Parts']
  Reddit: ['FedEx', "Love's Travel Stops", "McDonald's", 'NAPA Auto Parts']
  YouTube: ['FedEx', "Love's Travel Stops", "McDonald's", 'NAPA Auto Parts']
  News: ['FedEx', "Love's Travel Stops", "McDonald's", 'NAPA Auto Parts']


In [6]:
def create_merge_key(df, components=['race_number', 'sponsor']):
    df = df.copy()
    missing = [c for c in components if c not in df.columns]
    if missing:
        print(f'  Warning: columns missing for merge key: {missing}')
        return df
    df['merge_key'] = df[components].astype(str).agg('_'.join, axis=1)
    return df

race_df    = create_merge_key(race_df)
reddit_df  = create_merge_key(reddit_df)
youtube_df = create_merge_key(youtube_df)
news_df    = create_merge_key(news_df)

print('Sample merge keys (race results):')
print(race_df['merge_key'].head(5).tolist())

Sample merge keys (race results):
["1_McDonald's", '1_NAPA Auto Parts', '1_FedEx', "1_Love's Travel Stops", "2_McDonald's"]


In [7]:
import os
os.makedirs('../data/processed', exist_ok=True)
import json

race_df.to_csv('../data/processed/race_results_standardized.csv', index=False)
reddit_df.to_csv('../data/processed/reddit_mentions_standardized.csv', index=False)
youtube_df.to_csv('../data/processed/youtube_engagement_standardized.csv', index=False)
news_df.to_csv('../data/processed/news_mentions_standardized.csv', index=False)

standardization_log = {
    'date_format': DATE_FORMAT,
    'sponsor_name_mapping_count': len(SPONSOR_NAME_MAPPING),
    'merge_key_components': ['race_number', 'sponsor'],
    'target_sponsors': TARGET_SPONSORS,
    'fixes_applied': [
        'Reddit race_number 0 → 1 (Daytona pre-season mapped to Daytona 500)',
        'YouTube race_number -2 → 1 (Daytona 500 lookup fix)',
    ],
    'standardization_date': datetime.now().isoformat()
}
with open('../data/processed/standardization_log.json', 'w') as f:
    json.dump(standardization_log, f, indent=2)

print('Standardized datasets saved.')
print('Standardization log saved.')

Standardized datasets saved.
Standardization log saved.


## Step 3: Merge Race and Exposure Data

In [8]:
# Load standardized data
race_df    = pd.read_csv('../data/processed/race_results_standardized.csv')
reddit_df  = pd.read_csv('../data/processed/reddit_mentions_standardized.csv')
youtube_df = pd.read_csv('../data/processed/youtube_engagement_standardized.csv')
news_df    = pd.read_csv('../data/processed/news_mentions_standardized.csv')

# Create race-level base: one row per race_number + sponsor
race_base = race_df.groupby(['race_number', 'race_name', 'race_date', 'sponsor']).agg(
    driver         = ('driver',         'first'),
    team           = ('team',           'first'),
    finish_position = ('finish_position', 'min'),   # best finish across entries
    laps_led       = ('laps_led',        'sum')
).reset_index()

race_base['merge_key'] = race_base['race_number'].astype(str) + '_' + race_base['sponsor']

expected_races    = 36
expected_sponsors = len(TARGET_SPONSORS)
print(f'Base dataset: {len(race_base)} sponsor-race combinations')
print(f'Expected:     {expected_races} races x {expected_sponsors} sponsors = {expected_races * expected_sponsors}')
print(f'\nSponsor coverage:')
print(race_base.groupby('sponsor').size())

Base dataset: 144 sponsor-race combinations
Expected:     36 races x 4 sponsors = 144

Sponsor coverage:
sponsor
FedEx                  36
Love's Travel Stops    36
McDonald's             36
NAPA Auto Parts        36
dtype: int64


In [9]:
# --- Merge Reddit ---
reddit_merge = reddit_df[['merge_key', 'mention_count', 'total_score', 'avg_score',
                           'total_comments', 'engagement_per_mention']].copy()
reddit_merge = reddit_merge.rename(columns={
    'mention_count':         'reddit_mentions',
    'total_score':           'reddit_total_score',
    'avg_score':             'reddit_avg_score',
    'total_comments':        'reddit_total_comments',
    'engagement_per_mention':'reddit_engagement_per_mention',
})
dupes = reddit_merge['merge_key'].duplicated().sum()
if dupes > 0:
    print(f'Warning: {dupes} duplicate merge keys in Reddit data — keeping first')
    reddit_merge = reddit_merge.drop_duplicates(subset='merge_key', keep='first')

master_df = race_base.merge(reddit_merge, on='merge_key', how='left', indicator='_reddit_merge')
matched = (master_df['_reddit_merge'] == 'both').sum()
print(f'Reddit merge: {matched}/{len(master_df)} matched ({matched/len(master_df)*100:.1f}%)')

reddit_cols = ['reddit_mentions', 'reddit_total_score', 'reddit_avg_score',
               'reddit_total_comments', 'reddit_engagement_per_mention']
for col in reddit_cols:
    master_df[col] = master_df[col].fillna(0)
master_df = master_df.drop(columns=['_reddit_merge'])

Reddit merge: 104/144 matched (72.2%)


In [10]:
# --- Merge YouTube ---
youtube_merge = youtube_df[['merge_key', 'total_race_views', 'sponsor_video_count',
                             'sponsor_video_views', 'sponsor_video_likes', 'view_share']].copy()
youtube_merge = youtube_merge.rename(columns={
    'total_race_views':    'youtube_total_views',
    'sponsor_video_count': 'youtube_sponsor_videos',
    'sponsor_video_views': 'youtube_sponsor_views',
    'sponsor_video_likes': 'youtube_sponsor_likes',
    'view_share':          'youtube_view_share',
})
dupes = youtube_merge['merge_key'].duplicated().sum()
if dupes > 0:
    print(f'Warning: {dupes} duplicate merge keys in YouTube data — keeping first')
    youtube_merge = youtube_merge.drop_duplicates(subset='merge_key', keep='first')

master_df = master_df.merge(youtube_merge, on='merge_key', how='left', indicator='_youtube_merge')
matched = (master_df['_youtube_merge'] == 'both').sum()
print(f'YouTube merge: {matched}/{len(master_df)} matched ({matched/len(master_df)*100:.1f}%)')

youtube_cols = ['youtube_total_views', 'youtube_sponsor_videos',
                'youtube_sponsor_views', 'youtube_sponsor_likes', 'youtube_view_share']
for col in youtube_cols:
    master_df[col] = master_df[col].fillna(0)
master_df = master_df.drop(columns=['_youtube_merge'])

YouTube merge: 104/144 matched (72.2%)


In [11]:
# --- Merge News ---
news_merge = news_df[['merge_key', 'total_mentions', 'primary_mentions', 'weighted_mentions']].copy()
news_merge = news_merge.rename(columns={
    'total_mentions':    'news_total_mentions',
    'primary_mentions':  'news_primary_mentions',
    'weighted_mentions': 'news_weighted_mentions',
})
dupes = news_merge['merge_key'].duplicated().sum()
if dupes > 0:
    print(f'Warning: {dupes} duplicate merge keys in News data — keeping first')
    news_merge = news_merge.drop_duplicates(subset='merge_key', keep='first')

master_df = master_df.merge(news_merge, on='merge_key', how='left', indicator='_news_merge')
matched = (master_df['_news_merge'] == 'both').sum()
print(f'News merge: {matched}/{len(master_df)} matched ({matched/len(master_df)*100:.1f}%)')

news_cols = ['news_total_mentions', 'news_primary_mentions', 'news_weighted_mentions']
for col in news_cols:
    master_df[col] = master_df[col].fillna(0)
master_df = master_df.drop(columns=['_news_merge'])

News merge: 89/144 matched (61.8%)


In [12]:
# Spot-check: compare merged values against original sources
def spot_check_merge(master_df, race_number, sponsor, original_dfs):
    print(f"\n{'='*55}")
    print(f'SPOT CHECK: Race {race_number} — {sponsor}')
    print(f"{'='*55}")
    merged = master_df[(master_df['race_number'] == race_number) & (master_df['sponsor'] == sponsor)]
    if merged.empty:
        print('No matching record found.')
        return
    m = merged.iloc[0]
    mk = m['merge_key']
    print(f'Merge key: {mk} | Race: {m["race_name"]} | Driver: {m["driver"]} | Pos: {m["finish_position"]}')

    for src, col_check in [
        ('reddit',  ('reddit_mentions',  'mention_count')),
        ('youtube', ('youtube_sponsor_views', 'sponsor_video_views')),
        ('news',    ('news_total_mentions',   'total_mentions')),
    ]:
        master_val = m[col_check[0]]
        orig = original_dfs[src]
        orig_row = orig[orig['merge_key'] == mk]
        if not orig_row.empty and col_check[1] in orig_row.columns:
            orig_val = orig_row.iloc[0][col_check[1]]
            status = 'MATCH' if master_val == orig_val else 'MISMATCH'
            print(f'  {src}: master={master_val} | original={orig_val} → {status}')
        else:
            print(f'  {src}: no original record (filled with 0)')

original_dfs = {'reddit': reddit_df, 'youtube': youtube_df, 'news': news_df}
spot_check_merge(master_df, 1,  'FedEx',          original_dfs)
spot_check_merge(master_df, 18, 'NAPA Auto Parts', original_dfs)
spot_check_merge(master_df, 35, "McDonald's",      original_dfs)


SPOT CHECK: Race 1 — FedEx
Merge key: 1_FedEx | Race: Daytona | Driver: Denny Hamlin | Pos: 19
  reddit: master=7.0 | original=7 → MATCH
  youtube: master=0.0 | original=0 → MATCH
  news: master=2.0 | original=2 → MATCH

SPOT CHECK: Race 18 — NAPA Auto Parts
Merge key: 18_NAPA Auto Parts | Race: New Hampshire | Driver: Chase Elliott | Pos: 18
  reddit: master=7.0 | original=7 → MATCH
  youtube: master=0.0 | original=0 → MATCH
  news: master=1.0 | original=1 → MATCH

SPOT CHECK: Race 35 — McDonald's
Merge key: 35_McDonald's | Race: Martinsville | Driver: Bubba Wallace | Pos: 18
  reddit: no original record (filled with 0)
  youtube: no original record (filled with 0)
  news: master=1.0 | original=1 → MATCH


In [13]:
print('\n' + '='*55)
print('MASTER DATASET SUMMARY')
print('='*55)
print(f'Shape: {master_df.shape[0]} rows x {master_df.shape[1]} columns')
print(f'\nRace coverage:  {master_df["race_number"].nunique()} unique races')
print(f'Sponsor coverage: {master_df["sponsor"].nunique()} sponsors')
print(f'\nSponsor counts:')
print(master_df.groupby('sponsor').size())
print(f'\nExposure totals:')
print(f'  Reddit interest score sum:  {master_df["reddit_mentions"].sum():,.0f}')
print(f'  YouTube total race views:   {master_df["youtube_total_views"].sum():,.0f}')
print(f'  News total mentions:        {master_df["news_total_mentions"].sum():,.0f}')
print(f'\nMissing values: {master_df.isnull().sum().sum()}')

master_df.to_csv('../data/processed/master_dataset_merged.csv', index=False)

merge_stats = {
    'merge_date':         datetime.now().isoformat(),
    'total_rows':         len(master_df),
    'unique_races':       int(master_df['race_number'].nunique()),
    'unique_sponsors':    int(master_df['sponsor'].nunique()),
    'reddit_match_rate':  float((master_df['reddit_mentions'] > 0).mean()),
    'youtube_match_rate': float((master_df['youtube_total_views'] > 0).mean()),
    'news_match_rate':    float((master_df['news_total_mentions'] > 0).mean()),
}
with open('../data/processed/merge_statistics.json', 'w') as f:
    json.dump(merge_stats, f, indent=2)

print('\nMerged dataset saved: master_dataset_merged.csv')
print('Merge statistics saved: merge_statistics.json')


MASTER DATASET SUMMARY
Shape: 144 rows x 22 columns

Race coverage:  36 unique races
Sponsor coverage: 4 sponsors

Sponsor counts:
sponsor
FedEx                  36
Love's Travel Stops    36
McDonald's             36
NAPA Auto Parts        36
dtype: int64

Exposure totals:
  Reddit interest score sum:  1,068
  YouTube total race views:   1,433,844,976
  News total mentions:        97

Missing values: 0

Merged dataset saved: master_dataset_merged.csv
Merge statistics saved: merge_statistics.json


## Step 4: Validate Data Quality and Document Gaps

In [14]:
master_df = pd.read_csv('../data/processed/master_dataset_merged.csv')

print('='*55)
print('DATA COMPLETENESS CHECK')
print('='*55)

expected_races    = 36
expected_sponsors = 4
expected_rows     = expected_races * expected_sponsors
print(f'\nExpected: {expected_races} races x {expected_sponsors} sponsors = {expected_rows}')
print(f'Actual:   {len(master_df)}')
print(f'Difference: {len(master_df) - expected_rows}')

all_combinations = {(r, s) for r in range(1, 37) for s in TARGET_SPONSORS}
actual_combinations = set(zip(master_df['race_number'], master_df['sponsor']))
missing_combinations = all_combinations - actual_combinations

print(f'\n--- Missing Combinations ---')
if missing_combinations:
    print(f'Missing {len(missing_combinations)} combinations:')
    for combo in sorted(missing_combinations)[:10]:
        print(f'  Race {combo[0]}: {combo[1]}')
else:
    print('All expected combinations present.')

print(f'\n--- Column Completeness ---')
for col in master_df.columns:
    null_count = master_df[col].isnull().sum()
    if null_count > 0:
        print(f'  {col}: {null_count} nulls ({null_count/len(master_df)*100:.1f}%)')
    else:
        print(f'  {col}: Complete')

DATA COMPLETENESS CHECK

Expected: 36 races x 4 sponsors = 144
Actual:   144
Difference: 0

--- Missing Combinations ---
All expected combinations present.

--- Column Completeness ---
  race_number: Complete
  race_name: Complete
  race_date: Complete
  sponsor: Complete
  driver: Complete
  team: Complete
  finish_position: Complete
  laps_led: Complete
  merge_key: Complete
  reddit_mentions: Complete
  reddit_total_score: Complete
  reddit_avg_score: Complete
  reddit_total_comments: Complete
  reddit_engagement_per_mention: Complete
  youtube_total_views: Complete
  youtube_sponsor_videos: Complete
  youtube_sponsor_views: Complete
  youtube_sponsor_likes: Complete
  youtube_view_share: Complete
  news_total_mentions: Complete
  news_primary_mentions: Complete
  news_weighted_mentions: Complete


In [15]:
print('='*55)
print('DATA CONSISTENCY CHECK')
print('='*55)

# Finish position range
print('\n--- Finish Position Range ---')
print(f'Range: {master_df["finish_position"].min()} to {master_df["finish_position"].max()}')
invalid_pos = master_df[(master_df['finish_position'] < 1) | (master_df['finish_position'] > 45)]
print('Invalid positions:', len(invalid_pos))

# Laps led range
print('\n--- Laps Led Range ---')
neg_laps = master_df[master_df['laps_led'] < 0]
print(f'Negative laps: {len(neg_laps)}')
print(f'Range: {master_df["laps_led"].min()} to {master_df["laps_led"].max()}')

# YouTube view share
print('\n--- YouTube View Share ---')
invalid_share = master_df[(master_df['youtube_view_share'] < 0) | (master_df['youtube_view_share'] > 100)]
if len(invalid_share) > 0:
    print(f'WARNING: {len(invalid_share)} records with view share outside 0–100')
else:
    print(f'All view shares valid (range: {master_df["youtube_view_share"].min():.1f}–{master_df["youtube_view_share"].max():.1f})')

# Date range
print('\n--- Date Range ---')
master_df['race_date'] = pd.to_datetime(master_df['race_date'])
min_date = master_df['race_date'].min()
max_date = master_df['race_date'].max()
print(f'Date range: {min_date.date()} to {max_date.date()}')
out_of_range = master_df[(master_df['race_date'] < '2024-02-01') | (master_df['race_date'] > '2024-11-30')]
print(f'Out-of-range dates: {len(out_of_range)}')

DATA CONSISTENCY CHECK

--- Finish Position Range ---
Range: 1 to 38
Invalid positions: 0

--- Laps Led Range ---
Negative laps: 0
Range: 0 to 163

--- YouTube View Share ---
All view shares valid (range: 0.0–16.2)

--- Date Range ---
Date range: 2024-02-18 to 2024-11-10
Out-of-range dates: 0


In [16]:
print('='*55)
print('OUTLIER DETECTION')
print('='*55)

def detect_outliers(df, column, threshold=3):
    series = df[column].replace(0, np.nan).dropna()
    if len(series) < 3 or series.std() == 0:
        return pd.DataFrame()
    z_scores = (series - series.mean()) / series.std()
    outlier_idx = z_scores[abs(z_scores) > threshold].index
    outliers = df.loc[outlier_idx].copy()
    outliers['z_score'] = z_scores[outlier_idx]
    return outliers

metrics_to_check = ['reddit_mentions', 'reddit_total_score',
                    'youtube_total_views', 'youtube_sponsor_views', 'news_total_mentions']
outlier_summary = {}
for metric in metrics_to_check:
    outliers = detect_outliers(master_df, metric)
    print(f'\n--- {metric} ---')
    if len(outliers) > 0:
        print(f'Found {len(outliers)} outliers (z-score > 3):')
        print(outliers[['race_name', 'sponsor', metric, 'z_score']].to_string(index=False))
    else:
        print('No statistical outliers detected')
    outlier_summary[metric] = len(outliers)

print('\n--- Top 5 Reddit Interest Scores ---')
print(master_df.nlargest(5, 'reddit_mentions')[['race_name', 'sponsor', 'driver', 'finish_position', 'reddit_mentions']].to_string(index=False))

print('\n--- Top 5 YouTube Total Views ---')
print(master_df.nlargest(5, 'youtube_total_views')[['race_name', 'sponsor', 'youtube_total_views', 'youtube_sponsor_views']].to_string(index=False))

OUTLIER DETECTION

--- reddit_mentions ---
Found 8 outliers (z-score > 3):
   race_name             sponsor  reddit_mentions  z_score
    Michigan               FedEx             31.0 3.220315
    Michigan Love's Travel Stops             31.0 3.220315
    Michigan          McDonald's             31.0 3.220315
    Michigan     NAPA Auto Parts             31.0 3.220315
Watkins Glen               FedEx             30.0 3.064975
Watkins Glen Love's Travel Stops             30.0 3.064975
Watkins Glen          McDonald's             30.0 3.064975
Watkins Glen     NAPA Auto Parts             30.0 3.064975

--- reddit_total_score ---
No statistical outliers detected

--- youtube_total_views ---
Found 4 outliers (z-score > 3):
race_name             sponsor  youtube_total_views  z_score
  Daytona               FedEx          120760176.0 3.882893
  Daytona Love's Travel Stops          120760176.0 3.882893
  Daytona          McDonald's          120760176.0 3.882893
  Daytona     NAPA Auto Parts   

In [17]:
print('='*55)
print('CROSS-SOURCE VALIDATION')
print('='*55)

wins     = master_df[master_df['finish_position'] == 1]
non_wins = master_df[master_df['finish_position'] > 1]

print(f'\n--- Win vs Non-Win Exposure ---')
print(f'{"Metric":<35} {"Wins":>12} {"Non-Wins":>12} {"Ratio":>8}')
print('-' * 70)
for metric in ['reddit_mentions', 'youtube_sponsor_views', 'news_total_mentions']:
    w_avg  = wins[metric].mean()
    nw_avg = non_wins[metric].mean()
    ratio  = w_avg / nw_avg if nw_avg > 0 else float('inf')
    print(f'{metric:<35} {w_avg:>12.1f} {nw_avg:>12.1f} {ratio:>8.2f}x')

major_races = ['Daytona', 'Coca-Cola 600', 'Southern 500']
major_df   = master_df[master_df['race_name'].isin(major_races)]
regular_df = master_df[~master_df['race_name'].isin(major_races)]

print(f'\n--- Major vs Regular Race Exposure ---')
print(f'{"Metric":<35} {"Major":>12} {"Regular":>12} {"Ratio":>8}')
print('-' * 70)
for metric in ['reddit_mentions', 'youtube_total_views', 'news_total_mentions']:
    maj_avg = major_df[metric].mean() if len(major_df) > 0 else 0
    reg_avg = regular_df[metric].mean() if len(regular_df) > 0 else 0
    ratio   = maj_avg / reg_avg if reg_avg > 0 else float('inf')
    print(f'{metric:<35} {maj_avg:>12.1f} {reg_avg:>12.1f} {ratio:>8.2f}x')

CROSS-SOURCE VALIDATION

--- Win vs Non-Win Exposure ---
Metric                                      Wins     Non-Wins    Ratio
----------------------------------------------------------------------
reddit_mentions                              7.8          7.4     1.05x
youtube_sponsor_views                        0.0      68155.0     0.00x
news_total_mentions                          1.0          0.7     1.51x

--- Major vs Regular Race Exposure ---
Metric                                     Major      Regular    Ratio
----------------------------------------------------------------------
reddit_mentions                              3.5          7.6     0.46x
youtube_total_views                   60380088.0    6991207.9     8.64x
news_total_mentions                          1.2          0.6     1.95x


In [18]:
import os
os.makedirs('../docs', exist_ok=True)

gap_analysis = {
    'analysis_date': datetime.now().isoformat(),
    'dataset_summary': {
        'total_rows':     len(master_df),
        'unique_races':   int(master_df['race_number'].nunique()),
        'unique_sponsors': int(master_df['sponsor'].nunique()),
        'date_range':     f'{min_date.date()} to {max_date.date()}'
    },
    'completeness': {
        'missing_combinations': len(missing_combinations),
        'null_values': int(master_df.isnull().sum().sum())
    },
    'outliers': outlier_summary,
    'known_limitations': [
        'Reddit data sourced from Google Trends (pytrends) instead of Reddit API due to 403 blocks — interest_score (0-100) used as public interest proxy',
        'YouTube views counted at collection time (June 2026), not at race time (2024)',
        'News mentions limited to English-language sources via manual Google search',
        'Race dates approximated: spread evenly across Feb 18 – Nov 10 2024 (no Race_Date in source data)',
        'YouTube race_number -2 (Daytona) corrected to 1 during standardization',
        'YouTube coverage limited to races 1–34; races 35–36 have no YouTube data',
    ]
}

with open('../data/processed/gap_analysis.json', 'w') as f:
    json.dump(gap_analysis, f, indent=2, default=str)
print('Gap analysis saved.')

quality_report = f"""# Data Quality Report: NASCAR Sponsorship Master Dataset

**Generated**: {datetime.now().strftime('%Y-%m-%d %H:%M')}  
**Analyst**: Aylin Yasgul

---

## Dataset Overview

| Metric | Value |
|--------|-------|
| Total Records | {len(master_df):,} |
| Unique Races | {master_df['race_number'].nunique()} |
| Unique Sponsors | {master_df['sponsor'].nunique()} |
| Date Range | {min_date.date()} to {max_date.date()} |

## Completeness Assessment

- **Expected**: 36 races x 4 sponsors = 144 records
- **Actual**: {len(master_df)} records
- **Status**: {"COMPLETE" if len(master_df) == 144 else "GAPS IDENTIFIED"}

## Consistency Checks

| Check | Status | Notes |
|-------|--------|-------|
| Finish positions 1–45 | PASS | Range: {master_df['finish_position'].min()}–{master_df['finish_position'].max()} |
| Laps led non-negative | PASS | Range: {master_df['laps_led'].min()}–{master_df['laps_led'].max()} |
| Dates in 2024 season  | PASS | All dates within expected range |
| YouTube view share 0–100% | PASS | |

## Outlier Summary

Statistical outliers (z-score > 3):
- Reddit mentions: {outlier_summary.get('reddit_mentions', 0)} records
- YouTube views: {outlier_summary.get('youtube_total_views', 0)} records
- News mentions: {outlier_summary.get('news_total_mentions', 0)} records

Outliers are expected for major races and race wins.

## Known Limitations

1. **Reddit proxy**: Google Trends used instead of Reddit API (403 blocked). Interest score (0–100) is relative, not a raw mention count.
2. **YouTube timing**: Views counted at collection time (June 2026), not at race time.
3. **News depth**: Manual search; limited to ~10 results per race per sponsor.
4. **Race dates**: Approximated — source data has no Race_Date column.

## Recommendation

**Data quality is SUFFICIENT for analysis.** All critical consistency checks pass.

---
*NASCAR Sponsorship Visibility Analysis — Module 2.3*
"""

with open('../docs/data_quality_report.md', 'w') as f:
    f.write(quality_report)
print('Data quality report saved: docs/data_quality_report.md')

Gap analysis saved.
Data quality report saved: docs/data_quality_report.md


## Step 5: Finalize Master Dataset and Create Data Dictionary

In [19]:
master_df = pd.read_csv('../data/processed/master_dataset_merged.csv')

final_column_order = [
    # Identifiers
    'race_number', 'race_name', 'race_date', 'sponsor', 'driver', 'team',
    # Race Performance
    'finish_position', 'laps_led',
    # Reddit / Public Interest Exposure
    'reddit_mentions', 'reddit_total_score', 'reddit_avg_score',
    'reddit_total_comments', 'reddit_engagement_per_mention',
    # YouTube Exposure
    'youtube_total_views', 'youtube_sponsor_videos', 'youtube_sponsor_views',
    'youtube_sponsor_likes', 'youtube_view_share',
    # News Exposure
    'news_total_mentions', 'news_primary_mentions', 'news_weighted_mentions',
    # Merge key
    'merge_key'
]

# Keep only columns that exist (safe subset)
available = [c for c in final_column_order if c in master_df.columns]
master_df = master_df[available]

master_df = master_df.sort_values(['race_number', 'sponsor']).reset_index(drop=True)

# Final data type cleanup
master_df['race_number']     = master_df['race_number'].astype(int)
master_df['finish_position'] = master_df['finish_position'].astype(int)
master_df['laps_led']        = master_df['laps_led'].astype(int)
master_df['race_date']       = pd.to_datetime(master_df['race_date']).dt.strftime('%Y-%m-%d')

master_df.to_csv('../data/processed/master_dataset.csv', index=False)
print(f'Final master dataset saved: {len(master_df)} rows x {len(master_df.columns)} columns')
print(f'Columns: {list(master_df.columns)}')

Final master dataset saved: 144 rows x 22 columns
Columns: ['race_number', 'race_name', 'race_date', 'sponsor', 'driver', 'team', 'finish_position', 'laps_led', 'reddit_mentions', 'reddit_total_score', 'reddit_avg_score', 'reddit_total_comments', 'reddit_engagement_per_mention', 'youtube_total_views', 'youtube_sponsor_videos', 'youtube_sponsor_views', 'youtube_sponsor_likes', 'youtube_view_share', 'news_total_mentions', 'news_primary_mentions', 'news_weighted_mentions', 'merge_key']


In [20]:
race_count    = len(master_df['race_number'].unique())
reddit_count  = int((master_df['reddit_mentions'] > 0).sum())
youtube_count = int((master_df['youtube_total_views'] > 0).sum())
news_count    = int((master_df['news_total_mentions'] > 0).sum())

data_dictionary = f"""# Data Dictionary: NASCAR Sponsorship ROI Master Dataset

**File**: `master_dataset.csv`  
**Created**: {datetime.now().strftime('%Y-%m-%d')}  
**Records**: {len(master_df)} sponsor-race combinations  
**Coverage**: 2024 NASCAR Cup Series (Races 1–36)  
**Sponsors**: FedEx, NAPA Auto Parts, McDonald's, Love's Travel Stops

---

## Data Sources

| Source | Type | Collection Method | Records |
|--------|------|-------------------|---------|
| Race Results | Performance | Kaggle/Racing Reference | {len(master_df)} |
| Public Interest | Social Proxy | Google Trends (pytrends) | {reddit_count} |
| YouTube | Video | YouTube Data API v3 | {youtube_count} |
| News | Traditional Media | Manual Google Search | {news_count} |

**Note**: Public interest data uses Google Trends (search interest score 0–100) as a
proxy for Reddit engagement. Reddit's public JSON API returned HTTP 403 for all requests;
per mentor guidance, Google Trends was substituted as the social visibility signal.

---

## Column Definitions

### Identifier Columns

| Column | Type | Description | Example |
|--------|------|-------------|---------|
| `race_number` | integer | Sequential race number in 2024 season (1–36) | 1 |
| `race_name` | string | Abbreviated race/track name | "Daytona" |
| `race_date` | date (YYYY-MM-DD) | Approximate race date | "2024-02-18" |
| `sponsor` | string | Primary sponsor name (standardized) | "FedEx" |
| `driver` | string | Driver name | "Denny Hamlin" |
| `team` | string | Team name | "Joe Gibbs Racing" |
| `merge_key` | string | Unique join key (race_number_sponsor) | "1_FedEx" |

### Race Performance Columns

| Column | Type | Description | Range |
|--------|------|-------------|-------|
| `finish_position` | integer | Final finishing position | 1–40+ |
| `laps_led` | integer | Laps leading the race | 0–500+ |

**Notes**:
- Lower `finish_position` = better (1st place is best)
- `laps_led` indicates on-track visibility (car at front of field)

### Reddit / Public Interest Columns

| Column | Type | Description | Source |
|--------|------|-------------|--------|
| `reddit_mentions` | float | Weekly Google Trends interest score (0–100) | pytrends |
| `reddit_total_score` | float | Cumulative interest across weeks in race period | Calculated |
| `reddit_avg_score` | float | Average weekly interest in race period | Calculated |
| `reddit_total_comments` | integer | Placeholder (0 — Google Trends has no comment data) | N/A |
| `reddit_engagement_per_mention` | float | Average interest score per week | Calculated |

**Notes**:
- Values represent Google Trends relative search interest, not raw Reddit post counts
- 0 indicates no measurable interest in that race period
- Scores are relative within the 2024 season (100 = peak interest)

### YouTube Exposure Columns

| Column | Type | Description | Source |
|--------|------|-------------|--------|
| `youtube_total_views` | integer | Total views on all race highlight videos | YouTube API |
| `youtube_sponsor_videos` | integer | Count of videos featuring sponsor/driver | YouTube API |
| `youtube_sponsor_views` | integer | Total views on sponsor-relevant videos | YouTube API |
| `youtube_sponsor_likes` | integer | Total likes on sponsor-relevant videos | YouTube API |
| `youtube_view_share` | float | % of race views on sponsor videos | Calculated |

**Notes**:
- `total_views` represents race-level exposure; `sponsor_views` is sponsor-specific
- 0 = no videos found; YouTube data available for races 1–34 only
- View share = (sponsor_video_views / total_race_views) × 100

### News Exposure Columns

| Column | Type | Description | Source |
|--------|------|-------------|--------|
| `news_total_mentions` | integer | Total news articles mentioning sponsor | Manual Google search |
| `news_primary_mentions` | integer | Articles where sponsor was main focus | Manual classification |
| `news_weighted_mentions` | float | Mentions weighted by source tier | Calculated |

**Notes**:
- Source tiers: Tier 1 (ESPN, NASCAR.com) = 1.0 · Tier 2 = 0.8 · Tier 3 = 0.6 · Tier 4 = 0.4
- Primary vs secondary classification based on article headline and body focus

---

## Missing Value Treatment

All exposure metrics use **0** for missing values:
- Reddit: no measurable Google Trends interest in that period
- YouTube: no videos found for that sponsor/race
- News: no articles found in search

This is distinct from NULL, which would indicate data was not collected.

---

## Known Limitations

1. **Reddit proxy**: Google Trends used (not Reddit) — interest score is relative, not a raw count
2. **YouTube timing**: Views counted at collection time (June 2026), not at race time
3. **News depth**: Limited to ~10 results per search; may miss niche outlets
4. **Race dates**: Approximated linearly across season — source data has no Race_Date column

---

## Example Queries

```python
import pandas as pd
master_df = pd.read_csv('data/processed/master_dataset.csv')

# Filter to one sponsor
fedex_df = master_df[master_df['sponsor'] == 'FedEx']

# Find all race wins
wins = master_df[master_df['finish_position'] == 1]

# Total season exposure by sponsor
season_exposure = master_df.groupby('sponsor').agg({{
    'reddit_mentions':     'sum',
    'youtube_sponsor_views': 'sum',
    'news_total_mentions': 'sum'
}})

# Compare exposure: wins vs non-wins
wins     = master_df[master_df['finish_position'] == 1]
non_wins = master_df[master_df['finish_position'] > 1]
print(f"Avg Reddit interest — Wins: {{wins['reddit_mentions'].mean():.1f}}")
print(f"Avg Reddit interest — Non-wins: {{non_wins['reddit_mentions'].mean():.1f}}")

# Exposure trend over season
weekly = master_df.groupby('race_number').agg({{
    'reddit_mentions': 'sum',
    'youtube_total_views': 'sum'
}})
```

---

## File Manifest

| File | Location | Description |
|------|----------|-------------|
| `master_dataset.csv` | `data/processed/` | Final analysis-ready dataset |
| `data_dictionary.md` | `docs/` | This document |
| `data_quality_report.md` | `docs/` | Validation results |
| `gap_analysis.json` | `data/processed/` | Machine-readable limitations |
| `standardization_log.json` | `data/processed/` | Format standardization rules |
| `merge_statistics.json` | `data/processed/` | Merge operation summary |

---
*Last updated: {datetime.now().strftime('%Y-%m-%d')}*
"""

with open('../docs/data_dictionary.md', 'w') as f:
    f.write(data_dictionary)
print('Data dictionary saved: docs/data_dictionary.md')

print('\n=== Module 2.3 Complete ===')
print(f'master_dataset.csv:    {len(master_df)} rows x {len(master_df.columns)} columns')
print(f'data_dictionary.md:    docs/data_dictionary.md')
print(f'data_quality_report.md: docs/data_quality_report.md')

Data dictionary saved: docs/data_dictionary.md

=== Module 2.3 Complete ===
master_dataset.csv:    144 rows x 22 columns
data_dictionary.md:    docs/data_dictionary.md
data_quality_report.md: docs/data_quality_report.md
